<a href="https://colab.research.google.com/github/RodrigoYamaya/Projeto-spam-detector-IA/blob/main/Projeto_spam_detector_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub
import os
import pandas as pd

# 1. Iremos utilizar API DO  Kaggle para fazer os download dos dataset que sao os dados que irão alimentar nossa IA
path = kagglehub.dataset_download("venky73/spam-mails-dataset")
print("Caminho do dataset:", path)

# 2. Nesse techo vamos Pegar o arquivo CSV que ele acabou de baixar baixar via API, e garante que codigo localiza esse arquivo.
arquivos = os.listdir(path)
caminho_completo = os.path.join(path, arquivos[0])

#DATAFRAME: E transformar em dataframe(Dataframe e uma estrutura de dados bidimensional  semelhanre sql que organizamos a tabela linjas e colunas exatamente excel por exemplo.) a tabela e mostra as 5 primeiras linhas.

# 3. Nesse trecho vamos ler os arquivos de spam no csv atraves da biblioteca pandas,
df = pd.read_csv(caminho_completo)
df.head()

100%|██████████| 1.86M/1.86M [00:00<00:00, 77.6MB/s]

Extracting files...
Caminho do dataset: /root/.cache/kagglehub/datasets/venky73/spam-mails-dataset/versions/1


,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [3]:
# 1. Removemos as colunas inúteis (o índice velho e o texto do gabarito) que boa pratica ETL remover as colunas com dados inutil
df = df.drop(columns=['Unnamed: 0', 'label'])

# 2. Verificamos se existe alguma linha com valor vazio (Nulo/NaN)
print("Quantidade de valores vazios por coluna:")
print(df.isnull().sum())

# 3. Mostramos a nova cara da tabela
print("\nTabela limpa:")
df.head()

Quantidade de valores vazios por coluna:
text         0
label_num    0
dtype: int64

Tabela limpa:


,text,label_num
0,Subject: enron methanol ; meter # : 988291\r\n...,0
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,"Subject: photoshop , windows , office . cheap ...",1
4,Subject: re : indian springs\r\nthis deal is t...,0


In [4]:
import nltk
# otima biblioteca para extração do texto. E tambem linguagem natural que em resumo humana.
# stopwords e uma biblioteca que remove palavras que não acrescenta em  nada analise das nossa IA.Muito util na sermanticas das palavras "NLTk" implementa.
from nltk.corpus import stopwords
import string
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Baixa o dicionário de palavras inúteis em inglês
nltk.download('stopwords')

# A FUNÇÃO ENTRA AQUI: Ela ensina a IA a ignorar pontuação e palavras como "the", "is", "and"
def processaTexto(texto):
    nopunc = [char for char in texto if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    cleanWords = [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]
    return cleanWords

# 1). Separar texto e gabarito e dividir 80/20 que 80(treino(ensinar)) e os 20(teste(avaliar))
# o X(e a entrada dos nossos dados que seria o texto do email que conteudo)
# o Y(e o resultado correto da saida que gabarito da saida)
X = df['text']
y = df['label_num']
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)
#test_size=0.2(20% desses dados vao para teste), random_state=42
#codigo de cima(ele vai pegar nosso dataset e devide 80/20) treino e teste.


# 2). O PIPELINE: Ele junta a Vetorização, o cálculo matemático e o Treinamento num pacote só!
# pipeline vai criar um fluxo automatizado.
# 'vectorizer 'CountVectorizer conversao de texto para linguagem de maquina.Visto que IA NÃO entende texto humano(numeros,vetores,matematica,bytes) vindo NTLK
# TF-IDF ; E Usado para identificar a importancia das palavras.  ele pode reduzir peso em palavras comum e aulemtnar importancia certas palavras para assim identificar spam. Por exemplo palavra "FREE" "winner" são mais importancia para identicação.
# CLASSIFIER(MULTINOMINIALNB): MultinomialNB aki e o cerebro da IA que usar o algoritmo NAIVE BAYES(que algoritmo aprendizagem de maquina) que aprende probabilidades que o nome desse teorema bayes.  Por isso que ele usado para identificar spam. exemplo: se aparecer muito "winner", "free". ele começar assimilar que "SPAM"
pipeline = Pipeline([
    ('vectorizer', CountVectorizer(analyzer=processaTexto)), # Usa a função que criamos lá em cima
    ('tfidf', TfidfTransformer()),                           # Melhora o peso das palavras
    ('classifier', MultinomialNB())                          # Treina a IA
])



# 3. Treina e testa tudo de uma vez
# Previsoes(predict) :"IA, tente adivinhar os emails de teste" porque atraves do email de teste que ela nunca viu antes aprender e jogar direto no spam.
pipeline.fit(X_treino, y_treino)
previsoes = pipeline.predict(X_teste)
print(f"A precisão da nossa nova IA foi de: {accuracy_score(y_teste, previsoes) * 100:.2f}%")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


A precisão da nossa nova IA foi de: 91.98%


In [5]:
# Textos de teste em inglês que linguagem natural da IA
meus_emails = [
    "URGENT! You won a free mac book m5 pro max. Click the link to claim your prize now!",
    "Hi, attached is the financial report from today's meeting. Let me know if you have any questions."
]

# Passamos a lista de textos direto para o Pipeline
previsoes_novas = pipeline.predict(meus_emails)

print("--- TESTE DA NOSSA IA COM PIPELINE ---")
for email, previsao in zip(meus_emails, previsoes_novas):
    resultado = "SPAM (Lixo) 🚨" if previsao == 1 else "HAM (Normal) ✅"
    print(f"\nE-mail: '{email}'")
    print(f"Veredito da IA: {resultado}")

--- TESTE DA NOSSA IA COM PIPELINE ---

E-mail: 'URGENT! You won a free mac book m5 pro max. Click the link to claim your prize now!'
Veredito da IA: SPAM (Lixo) 🚨

E-mail: 'Hi, attached is the financial report from today's meeting. Let me know if you have any questions.'
Veredito da IA: HAM (Normal) ✅


In [6]:
import joblib

# Exportamos apenas o Pipeline QUE IREMOS USAR NA "API" QUE IREMOS EXPORTAR NO PROJETO.
# ARQUIVO PKL e um arquivo binario gerado python que usado para salvar objetos no codigo. Por exemplo modelos de inteligencia aritificial) diretamento nosso disco rigido.
joblib.dump(pipeline, 'pipeline_spam.pkl')

print("IA exportada com sucesso! O arquivo 'pipeline_spam.pkl' foi gerado.")

IA exportada com sucesso! O arquivo 'pipeline_spam.pkl' foi gerado.
